In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [2]:
df=pd.read_csv('email.csv')

In [3]:
mail_data=df.where((pd.notnull(df)),'')


In [4]:
mail_data.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
mail_data.shape

(5573, 2)

In [6]:
# Map textual labels to numeric (spam->0, ham->1) and drop invalid/missing rows
mail_data['Category'] = mail_data['Category'].map({'spam': 0, 'ham': 1})
print('Unique categories after mapping:', mail_data['Category'].unique())
invalid = mail_data['Category'].isnull().sum()
print('Rows with invalid/missing Category to drop:', invalid)
mail_data = mail_data[mail_data['Category'].notnull()].copy()
mail_data['Category'] = mail_data['Category'].astype(int)

Unique categories after mapping: [ 1.  0. nan]
Rows with invalid/missing Category to drop: 1


In [7]:
# Features (messages) and labels (0=spam,1=ham)
X = mail_data['Message'].fillna('').astype('str')
Y = mail_data['Category']


In [8]:
# Split into train/test (stratify to preserve class balance)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=3, stratify=Y)


In [9]:
print("X_train shape:",X_train.shape)
print("X_test shape:",X_test.shape)

X_train shape: (4457,)
X_test shape: (1115,)


In [10]:
# Feature extraction with TF-IDF
feature_extraction = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)
# Ensure messages are strings and replace NaNs (X was set earlier but do a final safeguard)
X_train = X_train.fillna('').astype('str')
X_test = X_test.fillna('').astype('str')
X_train_features = feature_extraction.fit_transform(X_train)
X_test_features = feature_extraction.transform(X_test)
# Ensure labels are integer dtype
Y_train = Y_train.astype('int')
Y_test = Y_test.astype('int')


In [11]:
print(X_train_features)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 34678 stored elements and shape (4457, 7493)>
  Coords	Values
  (0, 2353)	0.28101404009316056
  (0, 3806)	0.28101404009316056
  (0, 1857)	0.17073786814794129
  (0, 1193)	0.22908400928709988
  (0, 3113)	0.28101404009316056
  (0, 3627)	0.25144905621529934
  (0, 6330)	0.24059246244542992
  (0, 1540)	0.17407870571957915
  (0, 2644)	0.28101404009316056
  (0, 5029)	0.17467075796896542
  (0, 4306)	0.26793132631329497
  (0, 421)	0.25144905621529934
  (0, 4557)	0.28101404009316056
  (0, 6468)	0.26793132631329497
  (0, 1657)	0.28101404009316056
  (0, 0)	0.23628394623676158
  (1, 3982)	0.4167622750027118
  (1, 5911)	0.2761926296686631
  (1, 3941)	0.20702870014136815
  (1, 1969)	0.1749293187718031
  (1, 6553)	0.4722950153731612
  (1, 5947)	0.24356944504246256
  (1, 3948)	0.2761926296686631
  (1, 2113)	0.1985161464110967
  (1, 3828)	0.13684128003316173
  :	:
  (4456, 5091)	0.1743505991070133
  (4456, 7339)	0.13767285254208542
  (4456, 24

In [12]:
model = LogisticRegression()
model.fit(X_train_features, Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [13]:
prediction_on_training_data = model.predict(X_train_features)
accuracy_on_training_data = accuracy_score(Y_train, prediction_on_training_data)
print('Accuracy on training data : ', accuracy_on_training_data)

Accuracy on training data :  0.9667938074938299


In [14]:
prediction_on_test_data = model.predict(X_test_features)
accuracy_on_test_data = accuracy_score(Y_test, prediction_on_test_data)
print('Accuracy on test data : ', accuracy_on_test_data)

Accuracy on test data :  0.9713004484304932


In [15]:
input_mail = ["Congratulations! You've won a $1000 Walmart gift card. Click here to claim your prize."]
input_data_features = feature_extraction.transform(input_mail)
prediction = model.predict(input_data_features)
print(prediction)
if (prediction[0]==0):
    print('Spam mail')
else:
    print('Ham mail')

[0]
Spam mail
